# Trabajo Práctico: Introducción a Databricks y Procesamiento con Apache Spark
**Materia:** Análisis de Datos en la Nube  
**Profesor:** Luis A. Chiaramonte  
**Integrantes:** [Completar con Nombres de los Integrantes]  

---
### ¿De qué trata este trabajo?
El objetivo de este TP es dar nuestros primeros pasos en **Databricks Community Edition**, armando un pipeline ETL (Extracción, Transformación y Carga) completo con **Apache Spark**.

Vamos a:
1. Levantar un cluster y un notebook en Databricks.
2. Cargar dos tablas de un comercio electrónico (`ventas.csv` y `productos.csv`).
3. Limpiar los datos (sacar duplicados y emprolijar nulos).
4. Transformar y cruzar las tablas con un `join` para calcular métricas clave de facturación.
5. Guardar la tabla limpia en **Delta Lake** y validar que se pueda leer sin problemas.
6. Hacer consultas con **Spark SQL** y gráficos para sacar conclusiones del negocio.

## Etapa 1: Configuración del entorno
En Databricks la sesión de Spark ya viene lista y corriendo en segundo plano dentro de la variable global `spark`. Importamos las funciones que vamos a necesitar a lo largo del TP.

In [ ]:
# Importamos las funciones principales de PySpark SQL
from pyspark.sql.functions import col, round, sum, avg, count, when, desc
import matplotlib.pyplot as plt

# Verificamos qué versión de Spark está corriendo en el cluster
print(f"Sesión de Spark activa correctamente: Versión {spark.version}")

## Etapa 2: Carga y exploración de datos
Cargamos los dos archivos CSV que subimos previamente (a la carpeta local o a DBFS en Databricks). Le ponemos `header=True` para que use la primera fila como nombres de columna y `inferSchema=True` para que detecte automáticamente los números y textos.

In [ ]:
# Rutas de los archivos (si estás en Databricks podés usar 'dbfs:/FileStore/tables/ventas.csv')
path_ventas = "data/ventas.csv"
path_productos = "data/productos.csv"

# Leemos el archivo de ventas
df_ventas_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_ventas)

# Leemos el catálogo de productos
df_productos_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_productos)

print("Archivos cargados con éxito.")
print(f"Total de filas en ventas: {df_ventas_raw.count()}")
print(f"Total de productos: {df_productos_raw.count()}")

In [ ]:
# Miramos las primeras filas de cada DataFrame para ver cómo vienen los datos
print("--- Muestra de Ventas ---")
df_ventas_raw.show(5, truncate=False)

print("--- Muestra de Productos ---")
df_productos_raw.show(5, truncate=False)

In [ ]:
# Vemos los tipos de datos que infirió Spark
df_ventas_raw.printSchema()
df_productos_raw.printSchema()

## Etapa 3: Transformaciones y limpieza de datos
En esta etapa hacemos la parte fuerte del ETL:
1. **Manejo de nulos:** en `descuento` detectamos filas vacías. Asumimos que si no vino descuento, el valor es `0.0`.
2. **Eliminar duplicados:** encontramos filas repetidas que cargó el sistema dos veces por error.
3. **Join:** cruzamos las ventas con la tabla de productos usando `id_producto` para saber el nombre, categoría y precio de cada venta.
4. **Columnas calculadas:** calculamos el subtotal (`cantidad * precio_unitario`) y el total neto aplicando el descuento.

In [ ]:
# 1. Limpieza de duplicados
filas_antes = df_ventas_raw.count()
df_ventas_sin_dup = df_ventas_raw.dropDuplicates(["id_venta"])
filas_despues = df_ventas_sin_dup.count()

print(f"Filas antes: {filas_antes} | Filas después de eliminar duplicados: {filas_despues}")
print(f"Se eliminaron {filas_antes - filas_despues} registros duplicados.")

# 2. Rellenamos los descuentos nulos con 0.0
df_ventas_limpio = df_ventas_sin_dup.fillna({"descuento": 0.0})

In [ ]:
# 3. Hacemos el Join con la tabla de productos por id_producto
df_completo = df_ventas_limpio.join(
    df_productos_raw,
    on="id_producto",
    how="inner"
)

# 4. Creamos columnas calculadas para el negocio
# subtotal = cantidad * precio_unitario
# total_venta = subtotal - (subtotal * descuento)
df_transformado = df_completo \
    .withColumn("subtotal", round(col("cantidad") * col("precio_unitario"), 2)) \
    .withColumn("monto_descuento", round(col("subtotal") * col("descuento"), 2)) \
    .withColumn("total_venta", round(col("subtotal") - col("monto_descuento"), 2))

# Mostramos el resultado final de la transformación
df_transformado.select(
    "id_venta", "nombre_producto", "categoria", "cantidad",
    "precio_unitario", "descuento", "total_venta"
).show(10, truncate=False)

## Etapa 4: Almacenamiento con Delta Lake
Uno de los puntos clave de Databricks es **Delta Lake**. A diferencia de guardar en CSV o Parquet común, Delta nos da:
- **Transacciones ACID:** no se rompe la tabla si se cae el proceso a la mitad.
- **Time Travel:** podemos consultar versiones anteriores de la tabla.
- **Mayor velocidad:** gracias al registro transaccional y la indexación.

Acá guardamos la tabla transformada en formato Delta y después la leemos de vuelta para validar que no se perdió nada.

In [ ]:
# Guardamos en formato Delta Lake (modo overwrite para poder reejecutar sin problemas)
delta_path = "data/delta_ventas"

df_transformado.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print(f"Datos guardados exitosamente en Delta Lake en la ruta: {delta_path}")

In [ ]:
# Leemos nuevamente desde Delta Lake para validar la integridad
df_delta = spark.read.format("delta").load(delta_path)

print(f"Filas leídas desde Delta: {df_delta.count()}")
print("Esquema verificado en Delta Lake:")
df_delta.printSchema()

## Etapa 5: Consultas con Spark SQL, Visualización y Análisis
Para aprovechar la potencia de Spark SQL, registramos el DataFrame como una vista temporal (`vista_ventas`). Esto nos permite tirar consultas SQL tradicionales y armar gráficos.

In [ ]:
# Creamos una vista temporal para poder usar sintaxis SQL directa
df_delta.createOrReplaceTempView("vista_ventas")

# Consulta 1: Facturación total y cantidad de productos vendidos por Categoría
query_categorias = """
SELECT 
    categoria,
    COUNT(id_venta) AS operaciones,
    SUM(cantidad) AS unidades_vendidas,
    ROUND(SUM(total_venta), 2) AS facturacion_total
FROM vista_ventas
GROUP BY categoria
ORDER BY facturacion_total DESC
"""

df_resumen_cat = spark.sql(query_categorias)
df_resumen_cat.show(truncate=False)

In [ ]:
# Consulta 2: Top 3 productos que más dinero generaron
query_top_prod = """
SELECT 
    nombre_producto,
    categoria,
    SUM(cantidad) AS unidades_totales,
    ROUND(SUM(total_venta), 2) AS total_recaudado
FROM vista_ventas
GROUP BY nombre_producto, categoria
ORDER BY total_recaudado DESC
LIMIT 3
"""

df_top_prod = spark.sql(query_top_prod)
df_top_prod.show(truncate=False)

In [ ]:
# Visualización básica dentro del notebook (Gráfico de barras de ventas por categoría)
# En Databricks podés usar directamente: display(df_resumen_cat)
# Acá lo graficamos también con matplotlib para que quede visible en cualquier entorno:

pdf_cat = df_resumen_cat.toPandas()

plt.figure(figsize=(9, 4.5))
bars = plt.bar(pdf_cat["categoria"], pdf_cat["facturacion_total"] / 1e6, color=["#1E3A8A", "#E25822", "#0284C7", "#475569"])
plt.title("Facturación Total por Categoría (en Millones de Pesos)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Categoría", fontsize=11)
plt.ylabel("Millones de $", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f"${yval:.2f}M", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

## Etapa 6: Conclusiones del Trabajo

A partir del procesamiento y análisis de los datos en Databricks, llegamos a las siguientes conclusiones:

1. **Facilidad del entorno Databricks:** La plataforma resuelve toda la parte pesada de configurar máquinas y Spark. En pocos minutos tuvimos un cluster activo y pudimos combinar código Python con consultas SQL en el mismo notebook sin fricción.
2. **Importancia de la etapa de limpieza en el ETL:** Encontrar y eliminar duplicados y nulos antes de hacer los cálculos evitó que inflemos la facturación o tengamos errores de cálculo con los descuentos.
3. **Valor de Delta Lake:** Guardar en formato Delta nos garantiza que la tabla queda protegida con transacciones ACID, evitando tablas a medio escribir o datos corruptos ante cualquier falla del cluster.
4. **Hallazgo del negocio:** La categoría **Tecnología** es por lejos el motor de ingresos de la tienda (encabezada por las ventas de Notebooks Gamer), a pesar de que la categoría **Periféricos** tiene mayor cantidad de transacciones individuales.